# Eval Helper Test Experiment

Interactive mirror of `tests/test_eval.py` using file-backed dataset stages.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "nb":
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo))

from pra_core.datasets import load_dataset
from pra_torch.data import CharTokenizer
from pra_torch.eval import baseline_prompts, generated_contains_target

In [ ]:
tok = CharTokenizer(["abc <REF_1>"])
restored = CharTokenizer.from_vocab(tok.stoi)
text = "cab <REF_1>"

assert restored.decode(restored.encode(text)) == text
assert tok.encode("<REF_1>") == [tok.stoi["<REF_1>"]]
{"original_vocab": tok.vocab_size, "restored_vocab": restored.vocab_size, "ref_id": tok.stoi["<REF_1>"]}

In [ ]:
ex = load_dataset("stage0_synthetic_memory", repo / "data", max_examples=1)[0]
prompts = baseline_prompts(ex)

assert set(prompts) == {"no_refs", "full_context", "simple_rag", "pra"}
assert "Context:" in prompts["full_context"]
assert "Retrieved summary:" in prompts["simple_rag"]
assert prompts["pra"] == prompts["no_refs"]
prompts

In [ ]:
prompt = "Context has answer red-lynx-1. Answer:"

assert not generated_contains_target(prompt + " nope", prompt, "red-lynx-1")
assert generated_contains_target(prompt + " red-lynx-1", prompt, "red-lynx-1")
{
    "prompt_only_match": generated_contains_target(prompt + " nope", prompt, "red-lynx-1"),
    "generated_match": generated_contains_target(prompt + " red-lynx-1", prompt, "red-lynx-1"),
}